# Phase 6 Multi-Provider Comic Workflow Benchmark

This notebook now benchmarks 6 image models **through the comic workflow** instead of the plain image endpoint.

All runs call `POST /api/comics/generate-page`, so every provider receives the same:

- `storyPrompt`
- `styleId`
- `storyMemorySummary`
- `previousPages`
- `negativePrompt`
- comic-page system prompt assembly
- fixed 5-panel page layout instructions

The 6 benchmark runs are:

- OpenAI standard: `gpt-image-1.5`
- OpenAI fast: `gpt-image-1-mini`
- Gemini standard: `gemini-3-pro-image-preview`
- Gemini fast: `gemini-3.1-flash-image-preview`
- Doubao standard: `seedream-5-0-260128`
- Doubao fast: `seedream-5-0-lite-260128`

Before running it, start the 3.0 server from the repo root:

```powershell
cd "f:/Documents/GitHub/AI_TRPG_616/version 3.0"
npm.cmd run dev:server
```

For fairness, this notebook defaults to `REFERENCE_IMAGES = []`, because the current project only has fully wired reference-image transport for Gemini. If you later want to inspect character-consistency behavior, you can add PNG/JPG references manually and rerun.


In [ ]:
from __future__ import annotations

import base64
import json
import mimetypes
import os
import re
from base64 import b64decode
from datetime import datetime
from pathlib import Path
from time import perf_counter

import requests
from IPython.display import Image as IPyImage, SVG, display

repo_root = Path("f:/Documents/GitHub/AI_TRPG_616/version 3.0")
api_base_url = os.getenv("TRPG_IMAGE_NOTEBOOK_API_BASE_URL", "http://127.0.0.1:4316")
output_dir = repo_root / "artifacts" / "multi_provider_comic_benchmark"
output_dir.mkdir(parents=True, exist_ok=True)

print("Repo root:", repo_root)
print("API base URL:", api_base_url)
print("Output dir:", output_dir)


In [ ]:
OPENAI_IMAGE_COSTS = {
    ("gpt-image-1.5", "medium", "1024x1024"): 0.034,
    ("gpt-image-1-mini", "low", "1024x1024"): 0.005,
}

GEMINI_IMAGE_COSTS = {
    ("gemini-3-pro-image-preview", "1K"): 0.134,
    ("gemini-3.1-flash-image-preview", "1K"): 0.067,
}


def ensure_server_available() -> None:
    try:
        response = requests.get(f"{api_base_url}/api/health", timeout=10)
        response.raise_for_status()
    except Exception as exc:
        raise RuntimeError(
            f"AI TRPG 3.0 server is not reachable at {api_base_url}. "
            "Start it in a separate terminal with: cd 'f:/Documents/GitHub/AI_TRPG_616/version 3.0' ; npm.cmd run dev:server"
        ) from exc


def post_json(path: str, payload: dict) -> dict:
    response = requests.post(f"{api_base_url}{path}", json=payload, timeout=600)
    if not response.ok:
        raise RuntimeError(f"{response.status_code} {response.text}")
    return response.json()


def get_json(path: str) -> dict:
    response = requests.get(f"{api_base_url}{path}", timeout=60)
    if not response.ok:
        raise RuntimeError(f"{response.status_code} {response.text}")
    return response.json()


def decode_image_payload(image_url: str) -> tuple[bytes, str]:
    if image_url.startswith("data:"):
        header, encoded = image_url.split(",", 1)
        mime_type = header.split(";", 1)[0].replace("data:", "", 1)
        return b64decode(encoded), mime_type

    response = requests.get(image_url, timeout=300)
    response.raise_for_status()
    mime_type = response.headers.get("content-type", "image/png").split(";", 1)[0]
    return response.content, mime_type


def save_generated_image(image_url: str, output_stem: str) -> Path:
    payload, mime_type = decode_image_payload(image_url)
    extension = mimetypes.guess_extension(mime_type) or ".png"
    if mime_type == "image/svg+xml":
        extension = ".svg"
    if mime_type == "image/jpeg":
        extension = ".jpg"

    target_path = output_dir / f"{output_stem}{extension}"
    target_path.write_bytes(payload)
    return target_path


def display_generated_image(path: Path) -> None:
    if path.suffix.lower() == ".svg":
        display(SVG(filename=str(path)))
    else:
        display(IPyImage(filename=str(path)))


def make_output_stem(run_name: str) -> str:
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    safe_run_name = re.sub(r"[^a-zA-Z0-9_-]+", "-", run_name).strip("-").lower()
    return f"comic-workflow-benchmark-{safe_run_name}-{timestamp}"


def file_to_data_url(path: str | Path) -> str:
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"Image file not found: {file_path}")

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    raw = file_path.read_bytes()
    return f"data:{mime_type};base64,{base64.b64encode(raw).decode('ascii')}"


def normalize_previous_pages(previous_pages: list[dict]) -> list[dict]:
    normalized = []
    for page in previous_pages:
        item = {
            "pageNumber": int(page["pageNumber"]),
            "prompt": str(page["prompt"]),
        }
        if page.get("summary"):
            item["summary"] = str(page["summary"])
        image_path = page.get("image_path")
        image_url = page.get("imageUrl")
        if image_path:
            item["imageUrl"] = file_to_data_url(image_path)
        elif image_url:
            item["imageUrl"] = str(image_url)
        normalized.append(item)
    return normalized


def normalize_reference_images(reference_images: list[dict]) -> list[dict]:
    normalized = []
    for item in reference_images:
        image_path = item.get("image_path")
        image_url = item.get("imageUrl")
        normalized_item = {
            "role": item.get("role", "character")
        }
        if item.get("name"):
            normalized_item["name"] = str(item["name"])
        if item.get("appearance"):
            normalized_item["appearance"] = str(item["appearance"])
        if image_path:
            normalized_item["imageUrl"] = file_to_data_url(image_path)
        elif image_url:
            normalized_item["imageUrl"] = str(image_url)
        else:
            raise ValueError(f"Missing image_path/imageUrl in reference image item: {item}")
        normalized.append(normalized_item)
    return normalized


def estimate_cost(run: dict) -> tuple[float | None, str | None, str]:
    profile_id = run["imageProfileId"]
    config = run["runtimeImageModelConfig"]
    model = config.get("model")

    if profile_id == "chatgpt-image":
        key = (model, config.get("quality"), config.get("imageSize"))
        amount = OPENAI_IMAGE_COSTS.get(key)
        return amount, "USD" if amount is not None else None, "Estimated from OpenAI official image pricing for the configured model/quality/size."

    if profile_id == "gemini-image":
        key = (model, config.get("imageSize"))
        amount = GEMINI_IMAGE_COSTS.get(key)
        return amount, "USD" if amount is not None else None, "Estimated from Google Gemini official image pricing for the configured model/imageSize."

    return None, None, "No fixed per-image estimate is embedded for this provider run. Check the provider billing console for exact usage."


def format_cost(amount: float | None, currency: str | None) -> str:
    if amount is None or currency is None:
        return "unknown"
    return f"{amount:.4f} {currency}"


In [ ]:
# Change these fields to rerun the same comic workflow across all 6 models.

STYLE_ID = "manga"
NEGATIVE_PROMPT = "blurry unreadable text duplicated faces broken anatomy watermark bad hands extra fingers"

STORY_PROMPT = """Continue the story as page 2 of a supernatural subway mystery. Lin Yue, a rookie exorcist with a yellow raincoat over her school uniform, steps onto the flooded platform while Mori Akira, a tired rail engineer in a dark navy coat and white gloves, shines a maintenance lantern toward a ghost train arriving out of black water. Across five panels, show the train doors opening, rows of pale passengers turning to stare at them in unison, Lin gripping her prayer beads, Akira realizing the train number matches the line that vanished ten years ago, and the final panel ending on a close-up of one passenger holding Lin's childhood umbrella."""

PREVIOUS_PAGES = [
    {
        "pageNumber": 1,
        "prompt": "Lin Yue and Mori Akira descend into an abandoned midnight subway station where black water covers the tracks and old talismans float against the pillars.",
        "summary": "They hear an incoming train on a line that was erased from the city map ten years ago."
    }
]

STORY_MEMORY_SUMMARY = "Lin Yue is investigating a supernatural omen tied to missing commuters, while Mori Akira is the only surviving engineer who remembers the vanished line. The atmosphere should feel eerie, melancholic, and cinematic, with strong character continuity across panels."

# Keep empty for the fairest 6-model comparison. If you add refs, prefer PNG/JPG files.
REFERENCE_IMAGES = []
# Example:
# REFERENCE_IMAGES = [
#     {
#         "role": "character",
#         "name": "Lin Yue",
#         "appearance": "young East Asian woman, short black bob haircut, amber raincoat over a school uniform",
#         "image_path": repo_root / "artifacts" / "refs" / "lin-yue.png"
#     }
# ]

# Keep this as None to run all 6. Example: {"openai-standard", "gemini-fast"}
ENABLED_RUN_NAMES = None

IMAGE_RUNS = [
    {
        "runName": "openai-standard",
        "label": "OpenAI Standard",
        "imageProfileId": "chatgpt-image",
        "runtimeImageModelConfig": {
            "model": "gpt-image-1.5",
            "imageSize": "1024x1024",
            "quality": "medium",
            "outputFormat": "jpeg"
        },
        "docsUrl": "https://platform.openai.com/docs/guides/image-generation"
    },
    {
        "runName": "openai-fast",
        "label": "OpenAI Fast",
        "imageProfileId": "chatgpt-image",
        "runtimeImageModelConfig": {
            "model": "gpt-image-1-mini",
            "imageSize": "1024x1024",
            "quality": "low",
            "outputFormat": "jpeg"
        },
        "docsUrl": "https://platform.openai.com/docs/guides/image-generation"
    },
    {
        "runName": "gemini-standard",
        "label": "Gemini Standard",
        "imageProfileId": "gemini-image",
        "runtimeImageModelConfig": {
            "model": "gemini-3-pro-image-preview",
            "imageSize": "1K",
            "aspectRatio": "3:4"
        },
        "docsUrl": "https://ai.google.dev/gemini-api/docs/image-generation"
    },
    {
        "runName": "gemini-fast",
        "label": "Gemini Fast",
        "imageProfileId": "gemini-image",
        "runtimeImageModelConfig": {
            "model": "gemini-3.1-flash-image-preview",
            "imageSize": "1K",
            "aspectRatio": "3:4"
        },
        "docsUrl": "https://ai.google.dev/gemini-api/docs/image-generation"
    },
    {
        "runName": "doubao-standard",
        "label": "Doubao Standard",
        "imageProfileId": "doubao-image",
        "runtimeImageModelConfig": {
            "model": "seedream-5-0-260128",
            "imageSize": "2K",
            "watermark": False
        },
        "docsUrl": "https://docs.byteplus.com/en/docs/ModelArk/1824690"
    },
    {
        "runName": "doubao-fast",
        "label": "Doubao Fast",
        "imageProfileId": "doubao-image",
        "runtimeImageModelConfig": {
            "model": "seedream-5-0-lite-260128",
            "imageSize": "2K",
            "watermark": False
        },
        "docsUrl": "https://docs.byteplus.com/en/docs/ModelArk/1824690"
    }
]

active_runs = [
    run for run in IMAGE_RUNS
    if ENABLED_RUN_NAMES is None or run["runName"] in ENABLED_RUN_NAMES
]

print("Comic workflow benchmark loaded.")
print("Style:", STYLE_ID)
print("Prompt:", STORY_PROMPT)
print("Reference images:", len(REFERENCE_IMAGES))
print("Active runs:", [run["runName"] for run in active_runs])


In [ ]:
ensure_server_available()
presets = get_json("/api/comics/presets")
print("Available styles:", [style["id"] for style in presets["styles"]])
print("Page layout:", presets["pageLayout"])

normalized_previous_pages = normalize_previous_pages(PREVIOUS_PAGES)
normalized_reference_images = normalize_reference_images(REFERENCE_IMAGES)
results = []

for run in active_runs:
    print("\n" + "=" * 96)
    print(f"Running {run['label']} -> {run['runtimeImageModelConfig']['model']}")
    print(f"Docs: {run['docsUrl']}")

    payload = {
        "storyPrompt": STORY_PROMPT,
        "styleId": STYLE_ID,
        "storyMemorySummary": STORY_MEMORY_SUMMARY,
        "previousPages": normalized_previous_pages,
        "referenceImages": normalized_reference_images,
        "negativePrompt": NEGATIVE_PROMPT,
        "allowFallback": False,
        "imageProfileId": run["imageProfileId"],
        "runtimeImageModelConfig": run["runtimeImageModelConfig"]
    }

    started = perf_counter()
    try:
        response = post_json("/api/comics/generate-page", payload)
        elapsed = perf_counter() - started
        output_path = save_generated_image(response["imageUrl"], make_output_stem(run["runName"]))
        estimated_cost_amount, estimated_cost_currency, billing_note = estimate_cost(run)

        result_summary = {
            "runName": run["runName"],
            "label": run["label"],
            "profileId": run["imageProfileId"],
            "model": run["runtimeImageModelConfig"]["model"],
            "provider": response["provider"],
            "style": response["style"],
            "pageNumber": response["pageNumber"],
            "characterReferenceCount": response["characterReferenceCount"],
            "previousPageReferenceCount": response["previousPageReferenceCount"],
            "mimeType": response["mimeType"],
            "runtimeSeconds": round(elapsed, 3),
            "estimatedCost": format_cost(estimated_cost_amount, estimated_cost_currency),
            "savedImagePath": str(output_path),
            "billingNote": billing_note
        }
        print(json.dumps(result_summary, ensure_ascii=False, indent=2))
        print("\nFinal workflow prompt preview:\n")
        print(response["revisedPrompt"][:2200])
        if len(response["revisedPrompt"]) > 2200:
            print("\n...[prompt truncated for display]...")
        display_generated_image(output_path)
        results.append(result_summary)
    except Exception as exc:
        elapsed = perf_counter() - started
        failure_summary = {
            "runName": run["runName"],
            "label": run["label"],
            "profileId": run["imageProfileId"],
            "model": run["runtimeImageModelConfig"]["model"],
            "runtimeSeconds": round(elapsed, 3),
            "status": "failed",
            "error": str(exc)
        }
        print(json.dumps(failure_summary, ensure_ascii=False, indent=2))
        results.append(failure_summary)

print("\n" + "#" * 96)
print("Phase 6 comic workflow benchmark summary")
print(json.dumps(results, ensure_ascii=False, indent=2))


In [ ]:
import math

import matplotlib.pyplot as plt


def parse_estimated_cost(value: str | None) -> float | None:
    if not value or value == "unknown":
        return None
    try:
        return float(str(value).split()[0])
    except Exception:
        return None


if not results:
    raise RuntimeError("No benchmark results are available yet. Run the previous cell first.")

labels = [item.get("label", item.get("runName", "unknown")) for item in results]
runtime_values = [float(item.get("runtimeSeconds", 0.0) or 0.0) for item in results]
cost_values = [parse_estimated_cost(item.get("estimatedCost")) for item in results]
status_values = [item.get("status", "ok") for item in results]

runtime_colors = ["#d97706" if status == "failed" else "#2563eb" for status in status_values]
cost_colors = ["#9ca3af" if value is None else "#059669" for value in cost_values]
cost_plot_values = [0.0 if value is None else value for value in cost_values]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

runtime_bars = axes[0].barh(labels, runtime_values, color=runtime_colors)
axes[0].set_title("Phase 6 Runtime Comparison")
axes[0].set_xlabel("Seconds")
axes[0].invert_yaxis()
axes[0].grid(axis="x", linestyle="--", alpha=0.25)

for bar, runtime, status in zip(runtime_bars, runtime_values, status_values):
    suffix = " failed" if status == "failed" else ""
    axes[0].text(
        bar.get_width() + max(runtime_values + [1.0]) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{runtime:.2f}s{suffix}",
        va="center",
        fontsize=9,
    )

cost_bars = axes[1].barh(labels, cost_plot_values, color=cost_colors)
axes[1].set_title("Phase 6 Estimated Cost Comparison")
axes[1].set_xlabel("USD per image")
axes[1].invert_yaxis()
axes[1].grid(axis="x", linestyle="--", alpha=0.25)

known_cost_values = [value for value in cost_values if value is not None]
cost_text_offset = max(known_cost_values + [0.01]) * 0.03

for bar, cost_value in zip(cost_bars, cost_values):
    label_text = "unknown" if cost_value is None else f"${cost_value:.4f}"
    axes[1].text(
        bar.get_width() + cost_text_offset,
        bar.get_y() + bar.get_height() / 2,
        label_text,
        va="center",
        fontsize=9,
    )

failed_labels = [label for label, status in zip(labels, status_values) if status == "failed"]
if failed_labels:
    fig.suptitle("Phase 6 Comic Workflow Benchmark (failed runs shown in orange / gray)", fontsize=14)
    print("Failed runs:", failed_labels)
else:
    fig.suptitle("Phase 6 Comic Workflow Benchmark", fontsize=14)

plt.tight_layout()
plt.show()
